# 5.2 Region-based segmentation. K-means and GMM

In the previous notebook we had fun with boundary-based techniques for image segmentation. In this one we will play with region-based techniques, which form regions by grouping pixels according to their similarity, and where the resulting segments cover the entire image. Concretely we will address two popular region-based methods: 

- K-means (<a href="#521">section 5.2.1</a>)
- Gaussian Mixture Model (GMM) clustering, fitted with Expectation-Maximization (EM) (<a href="#522">section 5.2.2</a>)

## Problem context - Color quantization

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/color-quantization.jpg" width="800"/>
    <figcaption>Fig. 1. Example of color quantization: original image (left) and the same image with a reduced number of colors (right).</figcaption>
</center>$\\[5pt]$

Color quantization is the process of reducing the number of distinct colors in an image while preserving its color appearance as much as possible. It has many applications, like image compression (e.g. GIFs, which only support 256 colors!), [content-based image retrieval](https://en.wikipedia.org/wiki/Content-based_image_retrieval), edge detection and segmentation preprocessing, printing, medical imaging, etc. 

Image segmentation techniques can be used to achieve color quantization, let's see how it works!

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Color quantization in practice</font>  
> **Suggested prompt**  
> *In the context of computer vision, where is color quantization used today, and which methods are used for it besides K-means?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 05. Image segmentation'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
import scipy.stats as stats
from ipywidgets import interact, fixed, widgets
matplotlib.rcParams['figure.figsize'] = (10.0, 10.0)
images_path = './images/'

## 5.2.1 K-means <a id="521"></a>

As commented, region-based techniques try to group together pixels that are similar. Such an issue is often called the *clustering problem*. Different attributes can be used to decide if two pixels are similar or not: intensity, color, texture, spatial proximity (pixel location), etc.

The **K-means algorithm** is a region-based technique that, given a set of elements (image pixels in our case), makes $K$ clusters out of them. Thereby, it is a perfect technique for addressing color quantization, since our goal is to reduce the color palette of an image to a fixed number of colors $K$. Concretely, K-means partitions the data points $X = \{x_i\}$ in a given space (*e.g.* grayscale or RGB color representations) into $K$ clusters $C_j$, represented by their means (**centroids**) $\mu_j$, by minimizing the **within-cluster sum of squared distances**:

$$
\min_{\{C_j\},\{\mu_j\}} J = \sum_{j=1}^{K} \sum_{x_i \in C_j} \lVert x_i - \mu_j \rVert^2
$$

In our case, the data point $x_i$ is the **feature vector** $f(x,y)$ describing the $i^{th}$ pixel that, as mentioned, could include information like the pixel color, intensity, texture, etc. Thus, $\mu_j$ represents **the mean of the feature vectors** of the pixels in cluster $C_j$. 

Let's see how the K-means algorithm works in a color domain, where each pixel is represented in an $n$-dimensional feature space (*e.g.* grayscale images define a 1D feature space, while RGB images a 3D space):

1. Pick the number $K$, that is, the number of clusters in which the image will be segmented (e.g. number of colors).
2. Place $K$ centroids $\mu_j$ in the color space (e.g. randomly), these are the centers of the regions. 
3. Each pixel is assigned to the cluster with the closest centroid, hence creating new clusters. 

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/kmeans-step-1.png" width="400"/>
    <figcaption>Fig. 2. Example in a 2D space (e.g. the Cr and Cb bands of the YCrCb color space) with 3 clusters. Each point is assigned to its closest centroid</figcaption>
</center>$\\[5pt]$

4. Recompute each centroid as the mean of the $N_j$ points assigned to its cluster: $\mu_j = \frac{1}{N_j} \sum_{x_i \in C_j} x_i$. 

<center>
<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/kmeans-step-2.png" width="400" align="center"/>
    <figcaption>Fig. 3. Example of how the centroids evolve over time</figcaption>
</center>$\\[5pt]$

5. Repeat steps 3 and 4 until convergence, that is, some previously defined criterion is fulfilled (*e.g.* the centers of regions do not move, or a certain number of iterations is reached).$\\[10pt]$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/kmeans-step-3.png" width="400" align="center"/>
    <figcaption>Fig. 4. Final segmentation result</figcaption>
</center>$\\[5pt]$

This procedure is the same independently of the number of dimensions in the workspace. 

This technique presents a number of strengths and limitations:

- **Strengths:**
  - It is simple and efficient.
  - It converges to a local optimum (not necessarily the global minimum).
- **Limitations:**
  - High usage of memory.
  - $K$ must be specified.
  - Sensitive to initialization (initial position of centroids).
  - Sensitive to outliers.
  - Works best for compact, roughly spherical clusters in the feature space (because of the usage of the Euclidean distance).

Fig. 5 shows the whole procedure: at each iteration, the points are assigned to their closest centroid (step 3) and the centroids move to the mean of their points (step 4), so $J$ decreases until the centroids stop moving.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/kmeans_demo.gif" width="800"/>
    <figcaption>Fig. 5. K-means with $K=3$ in a 2D feature space. Left, the clusters and their centroids $\mu_j$; right, the value of $J$ after each step.</figcaption>
</center>$\\[5pt]$

### K-means toy example

Luckily for us, OpenCV defines a method that performs K-means: [`cv2.kmeans()`](https://docs.opencv.org/4.x/d5/d38/group__core__cluster.html#ga9a34dc06c6ec9460e90860f15bcd2f88), so let's see how it works!

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> K-means in OpenCV</font>  
> **Suggested prompt**  
> *I need to use OpenCV to cluster the pixels of an image with K-means through the `cv2.kmeans()` function. Do not write the full solution. Explain what its main parameters mean (e.g. the number of clusters, the termination criteria and the number of attempts).*

Let's take a look at a toy 1D K-means example in order to get familiar with it. The following function, `binarize_kmeans()`, binarizes an input `image` by executing the K-means algorithm, where `it` sets its maximum number of iterations. 

Note that the stopping criteria can be either:
- if a maximum number of iterations is reached, or 
- if the centroid moved less than a certain `epsilon` value in an iteration.

In [ ]:
def binarize_kmeans(image,it):
    """ Binarize an image using k-means.   

        Args:
            image: Input image
            it: Maximum number of K-means iterations
    """    
    
    # Set random seed for centroids 
    cv2.setRNGSeed(124)
    
    # Flatten image
    flattened_img = image.reshape((-1,1))
    flattened_img = np.float32(flattened_img)
    
    #Set epsilon
    epsilon = 0.2
    
    # Establish stopping criteria (either `it` iterations or moving less than `epsilon`)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, it, epsilon)
    
    # Set K parameter (2 for thresholding)
    K = 2
    
    # Call kmeans using random initial position for centroids
    _,label,center=cv2.kmeans(flattened_img,K,None,criteria,1,cv2.KMEANS_RANDOM_CENTERS)
    
    # Color resultant labels
    center = np.uint8(center) # Get center coordinates as unsigned integers   
    print(center)
    flattened_img = center[label.flatten()] # Get the color (center) assigned to each pixel
    
    # Reshape vector image to original shape
    binarized = flattened_img.reshape((image.shape))
    
    # Show resultant image
    plt.subplot(2,1,1)
    plt.title("Binarized image")
    plt.imshow(binarized, cmap='gray',vmin=0,vmax=255)
    
    # Show how original histogram has been segmented
    plt.subplot(2,1,2)
    plt.title("Segmented histogram")
    plt.hist([image[binarized==center[0]].ravel(), image[binarized==center[1]].ravel()],256,range=[0,256], color=["black","gray"],stacked=True)


As you can see, [`cv2.kmeans()`](https://docs.opencv.org/4.x/d5/d38/group__core__cluster.html#ga9a34dc06c6ec9460e90860f15bcd2f88) returns two relevant arguments:

- `label`: Integer array that stores the cluster index for every pixel.
- `center`: Matrix containing the cluster centroids (each row represents a different centroid).

**Attention to this!!!** It is also remarkable the first function argument, which represents the data for clustering: an array of $n$-dimensional points with float coordinates. Such array has the shape $n_{\text{samples}} \times n_{\text{features}}$, i.e., it has as many rows as samples (pixels in the image), and as many columns as features describing those samples (for example, if using the intensity of a pixel in a grayscale image, there is only one feature). For that, the code line `image.reshape((-1,1))` converts the initial grayscale image with dimensions $242\times 1133$ into a flattened version of it with dimension $274186\times 1$, that is, 274186 samples (or pixels) with only one feature, its intensity.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Reshaping arrays with NumPy</font>  
> **Suggested prompt**  
> *I am working in Python with images loaded with OpenCV, which are NumPy arrays (e.g. a grayscale image has shape `(rows, cols)`). Explain how the NumPy function `np.reshape()` works using `image.reshape((-1,1))` as an example: what shape it returns, what `-1` means in the new shape, and how to recover the original shape of the image afterwards. Do not write the full solution.*

Below it is provided an interactive code so you can play with `cv2.kmeans()` by calling it with different `it` values. 

*As you can see, if $K=2$ in a grayscale image, it is a binarization method that doesn't need to fix a manual threshold. We could have used it, for example, when dealing with the plate recognition problem!*

In [ ]:
matplotlib.rcParams['figure.figsize'] = (10.0, 10.0)

image = cv2.imread(images_path + 'plate.jpg',0)

interact(binarize_kmeans, image=fixed(image),it=(2,5,1));

Notice that for 1D spaces and not high-resolution images K-means is very fast! (it only needs a few iterations to converge). What happens if K-means is applied to color images (3D space) in order to get color quantization?  

Now that you know how K-means works, you can experimentally answer such question!  

###  **<span style="color:green"><b><i>ASSIGNMENT 1: Playing with K-means</i></b></span>**

Write a script that:
- applies K-means to `malaga.png` with different values for $K$: $K=4$, $K=8$ and $K=16$, setting `epsilon=0.2` and `it=10` as convergence criteria, and 
- shows, in a $2\times2$ subplot, the 3 resulting images along with the input one.

Notice that in this case we are using 3 features per pixel, their R, G and B values, that is, $x_i = [R_i,G_i,B_i]^T$, so the input data for the kmeans function has the dimensions $n_{\text{pixels}} \times 3$.

<font color='blue'>**Expected output:**  </font>

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/exercise1.png" width="800"/>

In [ ]:
# Assignment 1
matplotlib.rcParams['figure.figsize'] = (15.0, 12.0)

# Read RGB image
image = cv2.imread(images_path + "malaga.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Flatten image
flattened_img = image.reshape((None,None))
flattened_img = np.float32(flattened_img)

# Set criteria
it = None
epsilon = None
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, it, epsilon)

# Apply k-means. Keep the third argument as None!
_,label4,center4=cv2.kmeans(None,None,None,None,30,cv2.KMEANS_RANDOM_CENTERS)
_,label8,center8=cv2.kmeans(None,None,None,None,30,cv2.KMEANS_RANDOM_CENTERS)
_,label16,center16=cv2.kmeans(None,None,None,None,30,cv2.KMEANS_RANDOM_CENTERS)

# Color resultant labels
center4 = np.uint8(center4)
center8 = np.uint8(center8)
center16 = np.uint8(center16)

# Get the color (center) assigned to each pixel
res4 = center4[None]
res8 = center8[None]
res16 = center16[None]

# Reshape to original shape
quantized4 = res4.reshape((image.shape))
quantized8 = res8.reshape((image.shape))
quantized16 = res16.reshape((image.shape))

# Show original image
plt.subplot(2,2,1)
plt.title("Original image")
plt.imshow(None)

# Show k=4
plt.subplot(2,2,2)
plt.title("k=4")
plt.imshow(None)

# Show k=8
plt.subplot(2,2,3)
plt.title("k=8")
plt.imshow(None)

# Show k=16
plt.subplot(2,2,4)
plt.title("k=16")
plt.imshow(None);

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now, **answer the following questions**:

- What is `cv2.kmeans()` doing in each iteration?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- What number of maximum iterations did you use? Why?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- How could we compress these images so they require less space in memory? *Note: consider that a pixel in RGB needs 3 bytes to be represented, 8 bits per band.*
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

## 5.2.2 Gaussian Mixture Model (GMM) clustering with Expectation-Maximization (EM) <a id="522"></a>

**Gaussian Mixture Model (GMM) clustering** is the generalization of K-means where the feature distribution is represented as a **mixture of Gaussian clusters**. Each cluster $C_j \sim \mathcal{N}(\mu_j,\Sigma_j)$ is represented by a mean $\mu_j$ (location), a covariance matrix $\Sigma_j$ (shape/spread) and a prior weight $\pi_j = P(C_j)$, instead of just a centroid. It uses a *soft assignment*, since it doesn't give *hard* decisions where a pixel belongs or not to a cluster, but the probability $P(C_j|x_i)$ of that pixel belonging to each cluster $C_j$. The GMM parameters are estimated with the iterative **Expectation-Maximization (EM)** algorithm, so at each iteration not just the mean of each cluster is refined (as in K-means), but also its covariance matrix and its weight.

Before going into detail on the theory behind EM, it is worth seeing how GMM clustering performs in the car plate problem. OpenCV provides a class implementing the needed functionality for fitting a GMM with EM to segment an image, called [`cv2.ml.EM()`](https://docs.opencv.org/4.x/d1/dfb/classcv_1_1ml_1_1EM.html).

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> GMM clustering with OpenCV</font>  
> **Suggested prompt**  
> *I need to use OpenCV (Python) to segment an image with a Gaussian Mixture Model through the `cv2.ml.EM` class. Do not write the full solution. Explain how to set the number of clusters, the covariance matrix type (`em.setCovarianceMatrixType()`) and the termination criteria, what `em.trainEM()` returns, and how to get the means and covariances of the clusters (`em.getMeans()`, `em.getCovs()`).*

In [ ]:
matplotlib.rcParams['figure.figsize'] = (10.0, 10.0)
cv2.setRNGSeed(5)

# Define parameters
n_clusters = 2 
covariance_type = 0 # 0: Spherical covariance matrix. 1: Diagonal covariance matrix. 2: Full covariance matrix
n_iter = 10
epsilon = 0.2

# Create EM empty object
em = cv2.ml.EM_create()

# Set parameters
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, n_iter, epsilon)
em.setClustersNumber(n_clusters)
em.setCovarianceMatrixType(covariance_type)
em.setTermCriteria(criteria)

# Read grayscale image
image = cv2.imread(images_path + "plate.jpg",0)

# Flatten image
flattened_img = image.reshape((-1,1))
flattened_img = np.float32(flattened_img)

# Apply EM
_, _, labels, _ = em.trainEM(flattened_img)

# Reshape labels to image size (binarization)
binarized = labels.reshape((image.shape))

# Show binarized image
plt.subplot(2,1,1)
plt.title("Binarized image")
plt.imshow(binarized, cmap="gray")

# --------------- Gaussian visualization ---------------

plt.subplot(2,1,2)
plt.title("Gaussian distribution of each cluster")

# Get means and covs (for grayscale 1D both)
means = em.getMeans()
covs = em.getCovs()

# Get standard deviation as NumPy array
sigmas = np.sqrt(covs)
sigmas = sigmas[:,0,0]

# Cast list to NumPy array
means = np.array(means)[:,0]

# Plot Gaussians
x = np.linspace(0, 256, 100)
plt.plot(x, stats.norm.pdf(x, loc = means[0], scale = sigmas[0]))
plt.plot(x, stats.norm.pdf(x, loc = means[1], scale = sigmas[1]))
plt.legend(['Black Region', 'White Region'])

plt.show()

As you can see, although in OpenCV K-means is implemented as a method and EM as a class, they operate in a similar way. In the example above, we are segmenting a car plate into two clusters, and **each cluster is defined by a Gaussian distribution** (a Gaussian distribution for the black region, and another one for the white region). This is the basis of GMM clustering, **but how are the GMM parameters estimated**?  

EM estimates them with an iterative algorithm, summarized in Fig. 6:

- **Initialization:** the $K$ Gaussians are initialized, typically with random ($\mu_j$,$\Sigma_j$) and the same prior weight $\pi_j = 1/K$ for each cluster.
- **E-step (Expectation):** computes the responsibilities $\gamma_{ij} = P(C_j|x_i)$, that is, how much each Gaussian $j$ explains each point $x_i$ (a soft assignment).
- **M-step (Maximization):** re-estimates the mean $\mu_j$, covariance $\Sigma_j$ and weight $\pi_j$ of each cluster from all the points, weighting each one by its responsibility $\gamma_{ij}$.

Both steps are repeated until convergence. To get a hard segmentation, each pixel is finally assigned to the cluster with the highest $\gamma_{ij}$.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/gmm_em.png" width="1000"/>
    <figcaption>Fig. 6. GMM clustering fitted with EM: the model of each cluster, the E-step and the M-step.</figcaption>
</center>$\\[5pt]$

Fig. 7 shows the whole procedure in a 2D feature space, where the log-likelihood increases at each iteration until it converges.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/gmm_demo.gif" width="800"/>
    <figcaption>Fig. 7. GMM clustering fitted with EM, with $K=3$ in a 2D feature space. Left, the Gaussians (ellipses at 1 and 2 standard deviations) and their weights; right, the log-likelihood after each iteration.</figcaption>
</center>$\\[5pt]$


Doesn't it remind you of the K-means algorithm? **What is the difference between them?** 

The main difference is that K-means employs the **Euclidean distance** to measure how near a point is to a cluster. In GMM clustering we use a distance in which **each dimension is weighted** by the **covariance matrix** of each cluster, which is also called **Mahalanobis distance**. Furthermore, for K-means a point of data **belongs or not to** a cluster (hard assignment), whereas in GMM clustering a point of data has a higher or lower **probability** to belong to a cluster (soft assignment). The table below summarizes other differences:

|  | **K-means** | **GMM fitted with EM** |
|:---:|:---:|:---:|
| **Cluster representation** | Centroid $\mu_j$ | $\mu_j$, $\Sigma_j$, $\pi_j$ |
| **Cluster initialization** | Randomly select K means | Initialize K Gaussian distributions ($\mu_j$,$\Sigma_j$) and $\pi_j$ |
| **Assignment** | Nearest centroid | E-step: soft memberships $P(C_j \mid x_i)$ |
| **Update** | Recompute centroid $\mu_j$ | M-step: recompute $\mu_j$, $\Sigma_j$, $\pi_j$ |
| **Membership** | Hard | Soft (probabilistic) |

If you are still curious about EM, you can find [here](https://www.youtube.com/watch?v=REypj2sy_5U) a more detailed explanation.

### **<span style="color:green"><b><i>ASSIGNMENT 2: Color quantization with YCrCb color space</i></b></span>**

In the next example, color quantization is realized using the YCrCb color space instead of RGB. Recall that you have more info about such a space available in [Appendix 12.2 Color spaces](../Chapter%2012.%20Appendices/12.2%20Color%20spaces.ipynb). In this way, color quantization is only applied to the two color bands Cr and Cb, neglecting the grayscale one Y. 

Notice that in this case, the feature space has 2 dimensions, one for the Cr band, and another dimension for the Cb, hence the feature vector describing the $i^{th}$ pixel results $x_i = [Cr_i,Cb_i]^T$. 

Let's see how it works!

**What to do?** Test and understand the following code.

In [ ]:
# Assignment 2
matplotlib.rcParams['figure.figsize'] = (15.0, 15.0)
cv2.setRNGSeed(5)

# Define parameters

n_clusters = 3 # Don't modify this parameter for this exercise

covariance_type = 2 # 0: Spherical covariance matrix. 1: Diagonal covariance matrix. 2: Full covariance matrix
n_iter = 10
epsilon = 0.2

# Create EM empty object
em = cv2.ml.EM_create()

# Set parameters
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, n_iter, epsilon)
em.setClustersNumber(n_clusters)
em.setCovarianceMatrixType(covariance_type)
em.setTermCriteria(criteria)

# Read color image
image = cv2.imread(images_path + "malaga.png")

# Convert to YCrCb
image = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

# Take color bands (2 lasts)
color_bands = image[:,:,1:3]

# Flatten image
flattened_img = color_bands.reshape((-1,2))
flattened_img = np.float32(flattened_img)

# Apply EM
_, _, labels, _ = em.trainEM(flattened_img)

# Color resultant labels
centers = em.getMeans()
centers = np.uint8(centers)
res = centers[labels.flatten()]

# Reshape to original shape
color_bands = res.reshape((image.shape[0:2]) + (2,))

# Merge original first band with quantized color bands
quantized = np.zeros(image.shape)
quantized[:,:,0] = image[:,:,0]
quantized[:,:,[1,2]] = color_bands

# Cast to unsigned data type
quantized = np.uint8(quantized)

# Reconvert to RGB
quantized_rgb = cv2.cvtColor(quantized, cv2.COLOR_YCrCb2RGB)
image_rgb = cv2.cvtColor(image, cv2.COLOR_YCrCb2RGB)

# Show original image
plt.subplot(1,2,1)
plt.title("Original image")
plt.imshow(image_rgb)

# Show resultant image
plt.subplot(1,2,2)
plt.title("YCrCb quantized colors (3 colors)")
plt.imshow(quantized_rgb);

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Once you have understood the code above, **answer the following questions:**

- What are the dimensions of the means $\mu_j$ and the covariance matrices $\Sigma_j$?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What are the dimensions of the input to `trainEM()`? Why?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Why are the obtained results so good using only 3 clusters?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- What compression would be better in terms of space in memory, a compression of an RGB image where each band uses 16 different values instead of the original 256, or a 4-color compression in a YCrCb image? *Hint: consider the bits needed to codify such information. Hint 2: the grayscale band in YCrCb, that is, Y, is not compressed.*
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>   

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> K-means vs GMM</font>  
> **Suggested prompt**  
> *I have used K-means and GMM clustering for color quantization. Guide me, without giving me the answer, to reason when the soft assignments and covariance matrices of a GMM are worth their extra cost compared to K-means, and when K-means is enough.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

## Conclusion

Congratulations on getting this work done! You have learned:

- how K-means clustering works and how to use it,
- how GMM clustering works and how its parameters are estimated with the EM algorithm,
- how to carry out color quantization and the importance of color spaces in this context, and
- some basics for image compression.

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

You have used YCrCb in this notebook because you were already familiar with it. The truth is that, in color quantization matters, [Lab color space](https://en.wikipedia.org/wiki/CIELAB_color_space) is commonly used.  

Surf the internet for information about the Lab color space and then **answer the following questions**:

- How does Lab color space work?
- Why is it typically used for color quantization?

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>